# 📝 Notebook 11 — Natural Language Explanation Generator
Person 3

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, json

BASE_DIR = "/content/drive/MyDrive/deepfake-project"
EXPL_DIR = os.path.join(BASE_DIR, "outputs/explanations")
os.makedirs(EXPL_DIR, exist_ok=True)
print("✅ Ready.")

Mounted at /content/drive
✅ Ready.


In [2]:
# ✅ Template-based Explanation Generator
THRESHOLDS = {"Visual":0.50,"Temporal":0.55,"Audio":0.55,"Lip-Sync":0.45,"Frequency":0.50}

DESCRIPTIONS = {
    "Visual":    {"high":"Facial regions show texture artifacts, boundary blending errors, and GAN synthesis patterns.",
                  "low": "No significant visual manipulation artifacts detected."},
    "Temporal":  {"high":"Frame-to-frame analysis found irregular blinking frequency and motion instability.",
                  "low": "Facial motion appears temporally consistent and natural."},
    "Audio":     {"high":"Spectral analysis detected over-smoothed speech and frequency artifacts from neural voice synthesis.",
                  "low": "Audio characteristics are consistent with natural human speech."},
    "Lip-Sync":  {"high":"Significant audio–lip synchronization mismatch — mouth articulation does not match speech phonemes.",
                  "low": "Lip movements are well-synchronized with the audio."},
    "Frequency": {"high":"DCT/FFT analysis revealed GAN upsampling fingerprints in the frequency domain.",
                  "low": "No anomalous frequency-domain patterns detected."},
}

def generate_explanation(result, sync_timeline=None):
    scores  = result["modality_scores"]
    verdict = result["verdict"]
    conf    = result["confidence_pct"]
    attn    = result["attention_weights"]
    flagged = [m for m,s in scores.items() if (s<THRESHOLDS[m] if m=="Lip-Sync" else s>THRESHOLDS[m])]
    dominant= max(attn, key=attn.get)
    lines = [
        f"{'='*50}",
        f"  DEEPFAKE DETECTION REPORT",
        f"  VERDICT: {verdict}  |  Confidence: {conf}%",
        f"{'='*50}",
        f"  Primary evidence source: {dominant} (weight={attn[dominant]:.3f})",
        "", "── Modality Analysis ──",
    ]
    for mod in DESCRIPTIONS:
        s     = scores.get(mod, 0.5)
        ok    = (s < THRESHOLDS[mod]) if mod=="Lip-Sync" else (s >= THRESHOLDS[mod])
        flag  = "⚠ ANOMALY" if ok else "✔ OK"
        level = "high" if ok else "low"
        lines += [f"  [{flag}] {mod} (score={s:.4f})", f"    {DESCRIPTIONS[mod][level]}"]
    if sync_timeline:
        bad = [f"{t:.1f}s" for t,s in sync_timeline if s < 0.45][:5]
        if bad: lines += ["", f"  Lip-sync failures at: {', '.join(bad)}"]
    lines += ["", "── Summary ──"]
    if verdict == "FAKE":
        lines.append(
            f"  The video was classified as FAKE based on {', '.join(flagged[:3])} "
            f"inconsistencies. Cross-modal discrepancies confirm AI-generated manipulation. "
            f"Real videos maintain strict synchronization across all signal channels."
        )
    else:
        lines.append("  All modalities consistent with authentic, unmanipulated video.")
    lines.append("="*50)
    return "\n".join(lines)

print("✅ generate_explanation() defined.")

✅ generate_explanation() defined.


In [3]:
# ✅ Demo
demo_result = {
    "final_probability": 0.91,  "verdict": "FAKE",  "confidence_pct": 91.0,
    "modality_scores":   {"Visual":0.88,"Temporal":0.76,"Audio":0.82,"Lip-Sync":0.12,"Frequency":0.91},
    "attention_weights": {"Visual":0.18,"Temporal":0.14,"Audio":0.22,"Lip-Sync":0.32,"Frequency":0.14},
}
demo_sync = [(i*0.2, 0.3 + 0.4*(i%3==0)) for i in range(30)]
explanation = generate_explanation(demo_result, sync_timeline=demo_sync)
print(explanation)

# Save
with open(os.path.join(EXPL_DIR,"demo_explanation.txt"), "w") as f:
    f.write(explanation)
with open(os.path.join(EXPL_DIR,"demo_result.json"), "w") as f:
    json.dump(demo_result, f, indent=2)
print(f"\n✅ Saved to {EXPL_DIR}")

  DEEPFAKE DETECTION REPORT
  VERDICT: FAKE  |  Confidence: 91.0%
  Primary evidence source: Lip-Sync (weight=0.320)

── Modality Analysis ──
  [⚠ ANOMALY] Visual (score=0.8800)
    Facial regions show texture artifacts, boundary blending errors, and GAN synthesis patterns.
  [⚠ ANOMALY] Temporal (score=0.7600)
    Frame-to-frame analysis found irregular blinking frequency and motion instability.
  [⚠ ANOMALY] Audio (score=0.8200)
    Spectral analysis detected over-smoothed speech and frequency artifacts from neural voice synthesis.
  [⚠ ANOMALY] Lip-Sync (score=0.1200)
    Significant audio–lip synchronization mismatch — mouth articulation does not match speech phonemes.
  [⚠ ANOMALY] Frequency (score=0.9100)
    DCT/FFT analysis revealed GAN upsampling fingerprints in the frequency domain.

  Lip-sync failures at: 0.2s, 0.4s, 0.8s, 1.0s, 1.4s

── Summary ──
  The video was classified as FAKE based on Visual, Temporal, Audio inconsistencies. Cross-modal discrepancies confirm AI-gener